# Notebook para geração de tabela com justificativa e path das imagens
Autor: Suzane Carol de Lima  
Entrega: 05/03/26
## Objetivo:
Notebook de geração da tabela para integração com o PDI Digital

# Importações de bibliotecas

In [1]:
import boto3
import pandas as pd
from io import BytesIO
import os
from botocore.exceptions import ClientError
import json

# Declaração de constantes

In [3]:
BUCKET = 's3-ma-analytics-259755696523-sa-east-1'
CAMINHO_BASE = 'EADigital/PDI_Agulha/ID_85520_OS_6000685899_OL4_PAG3_PAG1_B/'
NOME_ABA_PLANILHA = 'enterramento'
REGIAO = 'sa-east-1'
DFS = []

# Funções Necessárias

In [2]:
def carregar_configuracao_aws(caminho_config: str) -> dict:
    """
    Carrega o arquivo de configuração AWS em formato JSON.

    Parâmetros:
        caminho_config (str): Caminho para o arquivo JSON de configuração.

    Retorno:
        dict: Dicionário com as configurações carregadas.
    """
    with open(caminho_config) as f:
        config = json.load(f)
    return config


def obter_cliente_s3(config: dict):
    """
    Cria e retorna um cliente S3 utilizando as credenciais do dicionário de configuração.
    
    Parâmetros:
        config (dict): Dicionário com as credenciais AWS. Deve conter o bloco 'pdi-digital' com
                       as chaves 'aws_access_key_id', 'aws_secret_access_key' e opcionalmente 'session_token'.
    
    Retorno:
        Cliente S3 autenticado via boto3.
    """
    if 'session_token' in config['pdi-digital']:
        print("vou usar")
        return boto3.client(
            's3',
            aws_access_key_id=config['pdi-digital']['aws_access_key_id'],
            aws_secret_access_key=config['pdi-digital']['aws_secret_access_key'],
            aws_session_token=config['pdi-digital']['session_token']
        )
    return boto3.client(
        's3',
        aws_access_key_id=config['pdi-digital']['aws_access_key_id'],
        aws_secret_access_key=config['pdi-digital']['aws_secret_access_key']
    )

def obter_paginador_list_objects(s3):
    """
    Retorna o paginador para a operação 'list_objects_v2' no S3.

    Parâmetros:
        s3: Cliente S3 autenticado.

    Retorno:
        Paginador configurado para 'list_objects_v2'.
    """
    return s3.get_paginator('list_objects_v2')

def listar_pastas_s3(paginador, bucket, caminho_base):
    """
    Retorna uma lista com os caminhos das pastas em um bucket S3 a partir de um caminho base.

    Parâmetros:
        paginador: Paginador do S3 configurado para 'list_objects_v2'.
        bucket: Nome do bucket S3.
        caminho_base: Caminho base (prefixo) para busca das pastas.

    Retorno:
        Lista de strings com os caminhos das pastas encontradas.
    """
    pastas = []
    for pagina in paginador.paginate(Bucket=bucket, Prefix=caminho_base, Delimiter='/'):
        for cp in pagina.get('CommonPrefixes', []):
            pastas.append(cp['Prefix'])
    return pastas

def processar_planilhas_s3(folders, s3, bucket, region, sheet_name):
    """
    Processa e unifica planilhas Excel armazenadas em diferentes pastas de um bucket S3.

    Parâmetros:
        folders (list): Lista de caminhos das pastas no bucket.
        s3: Cliente S3 autenticado.
        bucket (str): Nome do bucket S3.
        region (str): Região AWS do bucket.
        sheet_name (str): Nome da aba a ser lida em cada planilha.

    Retorno:
        None. Salva os arquivos unificados em Excel e CSV localmente.
    """
    dfs = []
    for folder in folders:
        key = folder + 'resultado_agent.xlsx'
        try:
            obj = s3.get_object(Bucket=bucket, Key=key)
            print(f'Planilha encontrada em: {folder}')
            df = pd.read_excel(BytesIO(obj['Body'].read()), sheet_name=sheet_name)
            df = df.rename(columns=lambda x: x.lower())
            if 'frame_id' in df.columns and 'justificativa_enterramento' in df.columns:
                df = df[['frame_id', 'justificativa_enterramento']]
                df = df.rename(columns={'justificativa_enterramento': 'justificativa'})
                df['imagem'] = (
                    f'https://{bucket}.s3.{region}.amazonaws.com/{folder}frames/' + df['frame_id'].astype(str) + '.jpg'
                )
                dfs.append(df)
        except Exception as e:
            print(f'Erro ao processar {key}: {e}')

    if dfs:
        df_geral = pd.concat(dfs, ignore_index=True)
        df_geral.to_excel('TABELA_UNIFICADA.xlsx', index=False)
        df_geral.to_csv('TABELA_UNIFICADA.csv', index=False)
        print('Planilhas unificadas com sucesso.')
    else:
        print('Nenhuma planilha processada.')

# Configurando acesso ao S3

In [4]:
config = carregar_configuracao_aws('configuracao_aws/config.json')

In [5]:
s3 = obter_cliente_s3(config)

vou usar


# Geração da tabela para o PDI digital

In [11]:
paginator = obter_paginador_list_objects(s3)

In [12]:
pastas = listar_pastas_s3(paginador, BUCKET, CAMINHO_BASE)

In [13]:
processar_planilhas_s3(pastas, s3, BUCKET, REGIAO, NOME_ABA_PLANILHA)

Planilha encontrada em: EADigital/PDI_Agulha/ID_85520_OS_6000685899_OL4_PAG3_PAG1_B/2024-03-29_181802_Ch4_00/
Planilha encontrada em: EADigital/PDI_Agulha/ID_85520_OS_6000685899_OL4_PAG3_PAG1_B/2024-03-29_183302_Ch4_01/
Planilha encontrada em: EADigital/PDI_Agulha/ID_85520_OS_6000685899_OL4_PAG3_PAG1_B/2024-03-29_184759_Ch4_02/
Planilha encontrada em: EADigital/PDI_Agulha/ID_85520_OS_6000685899_OL4_PAG3_PAG1_B/2024-03-29_190257_Ch4_03/
Planilha encontrada em: EADigital/PDI_Agulha/ID_85520_OS_6000685899_OL4_PAG3_PAG1_B/2024-03-29_191755_Ch4_04/
Planilha encontrada em: EADigital/PDI_Agulha/ID_85520_OS_6000685899_OL4_PAG3_PAG1_B/2024-03-29_193253_Ch4_05/
Planilha encontrada em: EADigital/PDI_Agulha/ID_85520_OS_6000685899_OL4_PAG3_PAG1_B/2024-03-29_194751_Ch4_06/
Planilha encontrada em: EADigital/PDI_Agulha/ID_85520_OS_6000685899_OL4_PAG3_PAG1_B/2024-03-29_201205_Ch4_00/
Planilha encontrada em: EADigital/PDI_Agulha/ID_85520_OS_6000685899_OL4_PAG3_PAG1_B/2024-03-29_202705_Ch4_01/
Planilha e